### Import

In [1]:
import os; import pandas as pd
pd.options.display.float_format = '{:.3f}'.format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
import numpy as np; import matplotlib.pyplot as plt
import gurobipy as gp; from gurobipy import GRB
from itertools import product; from tqdm import tqdm
import importlib
import functions_utils; import functions_data
import functions_optimize; import functions_eval
importlib.reload(functions_data); importlib.reload(functions_optimize)
importlib.reload(functions_eval); importlib.reload(functions_utils)
from functions_utils import *; from functions_data import *
from functions_optimize import *; from functions_eval import *
import time

S = 100
LEVEL = "high"
SEED = 3

generation_data, I, T = load_generation_data(date_filter="2022-07-18")
R, P_RT, K, K0, _, _, CRATE, DRATE = load_parameters(I, T, generation_data, S, LEVEL, SEED)
P_DA, P_PN = load_price_data(P_RT)
INEFF = 0.9

# MYP = np.zeros((I, T, S)) ; MYM = np.zeros((I, T, S)) ; MZC = np.zeros((I, T, S)) ; MZD = np.zeros((I, T, S)) ; MDP = np.zeros((I, T, S)) ; MDM = np.zeros((I, T, S))  
# R_max = np.zeros((I, T)) ; R_min = np.zeros((I, T)); R_avg = np.zeros((I, T))
# for i, t in product(range(I), range(T)):
#     R_max[i, t] = np.max(R[i, t, :])
#     R_min[i, t] = np.min(R[i, t, :])
#     R_avg[i, t] = np.mean(R[i, t, :])
    
#     for s in range(S): 
#         MYP[i, t, s] = MDP[i, t, s] = R[i, t, s] + INEFF * DRATE[i]
#         MZC[i, t, s] = min((1 / INEFF) * CRATE[i], R[i, t, s])
#         MZD[i, t, s] = INEFF * DRATE[i]
#         MYM[i, t, s] = MDM[i, t, s] = R_max[i, t] + INEFF * DRATE[i]
        # MYM[i, t, s] = MDM[i, t, s] = R[i, t, s] + INEFF * DRATE[i]

✅ 총 10개 파일을 불러왔습니다: 1033.csv, 1818.csv, 2502.csv, 2503.csv, 2634.csv, 2698.csv, 2816.csv, 545.csv, 665.csv, 690.csv
📊 데이터 Shape: I=10, T=24, S=100
✅ 시뮬레이션 초기화 완료: S=100, Randomness='high', Random Seed=3, M1=3721.01, M2=9383.93
   - 개별 K 값: [200. 200. 400. 600. 100. 600. 300. 200. 300. 200.]


### Individual Optimization (original)

In [2]:
m1 = gp.Model("individual")
m1.setParam("MIPGap", 1e-5)

x_ind = m1.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
yp_ind = m1.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp") ; ym_ind = m1.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym")
z_ind = m1.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z")
zc_ind = m1.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc") ; zd_ind = m1.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")

m1.update()

obj = (gp.quicksum(P_DA[t] * x_ind[i, t] for i in range(I) for t in range(T)) + 
       gp.quicksum((1/S) * (P_RT[t, s] * yp_ind[i, t, s] - P_PN[t, s] * ym_ind[i, t, s]) for i in range(I) for t in range(T) for s in range(S)))
m1.setObjective(obj, GRB.MAXIMIZE)

for i, t, s in product(range(I), range(T), range(S)):
    m1.addConstr(R[i, t, s] - x_ind[i, t] == yp_ind[i, t, s] - ym_ind[i, t, s] + zc_ind[i, t, s] - zd_ind[i, t, s])
    m1.addConstr(zd_ind[i, t, s]/INEFF <= z_ind[i, t, s])
    m1.addConstr(zd_ind[i, t, s]/INEFF <= DRATE[i])
    m1.addConstr(zc_ind[i, t, s]*INEFF <= K[i] - z_ind[i, t, s])
    m1.addConstr(zc_ind[i, t, s]*INEFF <= CRATE[i])
    m1.addConstr(z_ind[i, t, s] <= K[i])
    m1.addConstr(z_ind[i, t + 1, s] == z_ind[i, t, s] + INEFF * zc_ind[i, t, s] - zd_ind[i, t, s] / INEFF)

for i, s in product(range(I), range(S)): m1.addConstr(z_ind[i, 0, s] == K0[i])

m1.optimize()

if m1.status == GRB.OPTIMAL:
    x_ind = np.array([[x_ind[i, t].X for t in range(T)] for i in range(I)])
    yp_ind = np.array([[[yp_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; ym_ind = np.array([[[ym_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    zc_ind = np.array([[[zc_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; zd_ind = np.array([[[zd_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    z_ind = np.array([[[z_ind[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)]) ; OBJ_IND = m1.objVal
    # phi1_ind = np.array([[[phi1_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; phi2_ind = np.array([[[phi2_ind[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])

Set parameter Username
Set parameter LicenseID to value 2611964
Academic license - for non-commercial use only - expires 2026-01-20
Set parameter MIPGap to value 1e-05
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 25.1.0 25B78)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
MIPGap  1e-05

Optimize a model with 169000 rows, 121240 columns and 385000 nonzeros
Model fingerprint: 0xe9659e9a
Coefficient statistics:
  Matrix range     [9e-01, 1e+00]
  Objective range  [3e-02, 1e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e-01, 4e+03]
Presolve removed 97960 rows and 25940 columns
Presolve time: 0.15s
Presolved: 71040 rows, 95300 columns, 327160 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Ordering time: 0.82s

Barrier statistics:
 AA' NZ     : 1.482e+06
 Factor NZ  : 8.134e+06 (roughly 130 MB of memory)
 Factor Ops : 2.

In [3]:
header = (f"{'t':>2} | {'R':>8} {'x':>8} {'y+':>8} {'y-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n" + "-" * 70)
print("\n[Individual]") ; print(header)
for t in range(7, 22):
    # R_avg = np.mean([R[:, t, s].sum() for s in range(S)]) ; x_sum = x_ind[:, t].sum()
    # yp_avg = np.mean([yp_ind[:, t, s].sum() for s in range(S)]) ; ym_avg = np.mean([ym_ind[:, t, s].sum() for s in range(S)])
    # zc_avg = np.mean([zc_ind[:, t, s].sum() for s in range(S)]) ; zd_avg = np.mean([zd_ind[:, t, s].sum() for s in range(S)]) 
    # z_avg = np.mean([z_ind[:, t, s].sum() for s in range(S)])

    i=1
    R_avg = np.mean([R[i, t, s] for s in range(S)]) ; x_sum = x_ind[i, t]
    yp_avg = np.mean([yp_ind[i, t, s] for s in range(S)]) ; ym_avg = np.mean([ym_ind[i, t, s] for s in range(S)])
    zc_avg = np.mean([zc_ind[i, t, s] for s in range(S)]) ; zd_avg = np.mean([zd_ind[i, t, s] for s in range(S)]) 
    z_avg = np.mean([z_ind[i, t, s] for s in range(S)])

    print(f"{t:>2} | {R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}")


[Individual]
 t |        R        x       y+       y-       zc       zd        z
----------------------------------------------------------------------
 7 |    77.72     9.71    28.32     0.00    39.82     0.13    31.65
 8 |   133.63    93.31    20.24     1.79    30.46     8.59    67.34
 9 |   168.47   126.77    28.18     4.90    28.76    10.33    85.21
10 |   177.17   147.80    25.36     8.72    27.02    14.29    99.61
11 |   232.51   186.50    43.39    11.20    28.08    14.26   108.05
12 |   216.98   161.38    55.30     5.03    21.62    16.28   117.48
13 |   499.76     0.00   540.82     0.00     0.94    42.00   118.85
14 |   556.21   179.17   370.34     0.00    19.29    12.60    73.03
15 |   463.06   419.97    96.07    60.72    27.12    19.39    76.40
16 |   169.57   169.33    10.74    12.91    20.98    18.57    79.26
17 |   197.25   182.09    24.57    12.77    20.32    16.96    77.51
18 |   268.54   225.30    61.45    20.16    19.89    17.93    76.95
19 |   105.56    94.22    19.16

### Holistic Optimization (Linear Decision Rule + MILP - M) (original)

In [ ]:
m2 = gp.Model("holistic_MILP_M")
m2.setParam("MIPGap", 1e-5)
m2.setParam(GRB.Param.TimeLimit, 1200)

x_hol = m2.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x") ; yp_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp") ; ym_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym")
dp_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp") ; dm_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm")
z_hol = m2.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z") ; zc_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc") ; zd_hol = m2.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")

m2.update()

obj_lin = gp.quicksum(
    P_DA[t] * x_hol[i, t] for i in range(I) for t in range(T)
) + gp.quicksum(
    (1 / S) * (P_RT[t, s] * yp_hol[i, t, s] - P_PN[t, s] * ym_hol[i, t, s])
    for i in range(I)
    for t in range(T)
    for s in range(S)
)

eps = 1e-8
# quad_reg = gp.quicksum(x_hol[i, t] * x_hol[i, t] for i in range(I) for t in range(T))
quad_reg = gp.quicksum((1 / S) * (dp_hol[i, t, s] * dp_hol[i, t, s] + dm_hol[i, t, s] * dm_hol[i, t, s]) for i in range(I) for t in range(T) for s in range(S))
obj = obj_lin - eps * quad_reg

# NOTE
m2.setObjective(obj_lin, GRB.MAXIMIZE)

for i, t, s in product(range(I), range(T), range(S)):
    m2.addConstr(R[i, t, s] - x_hol[i, t] == yp_hol[i, t, s] - ym_hol[i, t, s] + dp_hol[i, t, s] - dm_hol[i, t, s] + zc_hol[i, t, s] - zd_hol[i, t, s])
    m2.addConstr(zd_hol[i, t, s]/INEFF <= z_hol[i, t, s])
    m2.addConstr(zd_hol[i, t, s]/INEFF <= DRATE[i])
    m2.addConstr(zc_hol[i, t, s]*INEFF <= K[i] - z_hol[i, t, s])
    m2.addConstr(zc_hol[i, t, s]*INEFF <= CRATE[i])
    m2.addConstr(z_hol[i, t, s] <= K[i])
    m2.addConstr(z_hol[i, t + 1, s] == z_hol[i, t, s] + INEFF * zc_hol[i, t, s] - zd_hol[i, t, s] / INEFF)
for i, s in product(range(I), range(S)): m2.addConstr(z_hol[i, 0, s] == K0[i])

# NOTE : inefficiency 추가
balance_constraints = {}
INEFF_INT = 0.999
for t, s in product(range(T), range(S)):
    balance_constraints[t, s] = m2.addConstr(INEFF_INT * gp.quicksum(dp_hol[i, t, s] for i in range(I)) == gp.quicksum(dm_hol[i, t, s] for i in range(I)), name=f"balance_{t}_{s}")

m2.optimize()

if m2.status == GRB.OPTIMAL or m2.status == GRB.TIME_LIMIT:
    x_hol = np.array([[x_hol[i, t].X for t in range(T)] for i in range(I)])
    yp_hol = np.array([[[yp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; ym_hol = np.array([[[ym_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    dp_hol = np.array([[[dp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; dm_hol = np.array([[[dm_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    zc_hol = np.array([[[zc_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; zd_hol = np.array([[[zd_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    z_hol = np.array([[[z_hol[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)]) ; original_objval = m2.objVal
    OBJ_HOL = sum(P_DA[t] * x_hol[i, t] for i in range(I) for t in range(T)) + sum(
        (1 / S) * (P_RT[t, s] * yp_hol[i, t, s] - P_PN[t, s] * ym_hol[i, t, s])
        for i in range(I)
        for t in range(T)
        for s in range(S)
    )
    QUAD_HOL = eps * sum((1 / S) * (dp_hol[i, t, s] * dp_hol[i, t, s] + dm_hol[i, t, s] * dm_hol[i, t, s]) for i in range(I) for t in range(T) for s in range(S))

    lambda_dual = {}
    for t, s in product(range(T), range(S)): 
        lambda_dual[t, s] = balance_constraints[t, s].Pi
    print("Direct dual extraction successful!")

else:
    print(f"⚠️ Model finished with status: {m2.status}")

Set parameter MIPGap to value 1e-05
Set parameter TimeLimit to value 1200
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 25.1.0 25B78)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  1200
MIPGap  1e-05

Optimize a model with 171400 rows, 169240 columns and 481000 nonzeros
Model fingerprint: 0x3d74fcc8
Coefficient statistics:
  Matrix range     [9e-01, 1e+00]
  Objective range  [3e-02, 1e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e-01, 4e+03]
Presolve removed 96000 rows and 22010 columns
Presolve time: 0.18s
Presolved: 75400 rows, 147230 columns, 431000 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Ordering time: 1.98s
Ordering time: 2.01s

Barrier statistics:
 Dense cols : 230
 AA' NZ     : 4.460e+05
 Factor NZ  : 2.016e+06 (roughly 100 MB of memory)
 Factor Ops : 1.755e+08 (less than 1 second per iterat

In [263]:
QUAD_HOL

np.float64(0.03359349768018278)

In [264]:
header = (f"{'t':>2} | {'R':>8} {'x':>8} {'y+':>8} {'y-':>8} {'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n" + "-" * 90)
print(f"\n[HOLISTIC] Objective Value = {OBJ_HOL:.2f}") ; print(header)
for t in range(0, 24):
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)]) ; x_sum = x_hol[:, t].sum()
    yp_avg = np.mean([yp_hol[:, t, s].sum() for s in range(S)]) ; ym_avg = np.mean([ym_hol[:, t, s].sum() for s in range(S)])
    dp_avg = np.mean([dp_hol[:, t, s].sum() for s in range(S)]) ; dm_avg = np.mean([dm_hol[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_hol[:, t, s].sum() for s in range(S)]) ; zd_avg = np.mean([zd_hol[:, t, s].sum() for s in range(S)]) ; z_avg = np.mean([z_hol[:, t, s].sum() for s in range(S)])
    print(f"{t:>2} | {R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} {INEFF_INT * dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}")


[HOLISTIC] Objective Value = 5569728.55
 t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
 0 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00
 1 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00
 2 |     0.00     0.00     0.00     5.37     0.00     0.00     5.37     0.00     0.00
 3 |     0.00     0.00     0.00    58.92     0.00     0.00    58.92     0.00     4.83
 4 |     0.00     0.00     0.00   163.05     0.00     0.00   163.05     0.00    57.86
 5 |     0.00     0.00     0.00     3.52     0.00     0.00     3.52     0.00   204.60
 6 |    56.77     0.00     6.64     0.00     4.76     4.76    50.13     0.00   207.78
 7 |   595.74     0.00    80.41     0.00   130.92   130.92   515.96     0.77   252.90
 8 |  1451.63   782.59    27.60     0.00   200.99   200.99   641.23     0.00   716.41
 9 |  15

In [265]:
for i, t, s in product(range(I), range(T), range(S)):
    if dp_hol[i, t, s] > 0.001 and dm_hol[i, t, s] > 0.001:
        print(f"i={i}, t={t}, s={s}, dp={dp_hol[i, t, s]}, dm={dm_hol[i, t, s]}")
    if zc_hol[i, t, s] > 0.001 and zd_hol[i, t, s] > 0.001:
        print(f"i={i}, t={t}, s={s}, zc={zc_hol[i, t, s]}, zd={zd_hol[i, t, s]}")
    if dp_hol[i, t, s] > 0.001 and ym_hol[i, t, s] > 0.001:
        print(f"i={i}, t={t}, s={s}, dp={dp_hol[i, t, s]}, ym={ym_hol[i, t, s]}, P_RT={P_RT[t, s]}, P_IN={-lambda_dual[t, s] * S}, P_PN={P_PN[t, s]}")
    if dm_hol[i, t, s] > 0.001 and yp_hol[i, t, s] > 0.001:
        print(f"i={i}, t={t}, s={s}, dm={dm_hol[i, t, s]}, yp={yp_hol[i, t, s]}, P_RT={P_RT[t, s]}, P_IN={-lambda_dual[t, s] * S}, P_PN={P_PN[t, s]}")

### Individual Replay

In [266]:
lambda_rep = np.zeros((T, S))
data = []
# eps = 0.00001
eps = 0

for t in range(T):
    for s in range(S):
        lambda_rep[t, s] = lambda_dual[t, s]
        # lambda_rep[t,s] = np.clip(lambda_dual[t,s], -P_PN[t,s] / S, -P_RT[t,s] / S)
        # lambda_rep[t, s] = ((-P_RT[t, s] / S) + (-P_PN[t, s] / S))/2
        
        if abs(lambda_rep[t, s] - (-P_RT[t, s] / S)) < 0.001:
            lambda_rep[t, s] -= eps / S
            
        elif abs(lambda_rep[t, s] - (-P_PN[t, s] / S)) < 0.001:
            lambda_rep[t, s] += eps / S
    
    s_fixed = 0
    data.append({
        'Time': t, 
        'P_DA': round(P_DA[t], 2), 
        'P_RT_avg': round(P_RT[t, s_fixed], 5), 
        'Lambda': round(-lambda_rep[t, s_fixed] * S, 5), 
        'P_PN_avg': round(P_PN[t, s_fixed], 4)
    })

pd.DataFrame(data)

# data_for_csv = []

# for t in range(T):
#     for s in range(S):
#         row = {
#             "t": t,
#             "s": s,
#             "P_DA": P_DA[t],
#             "P_RT": P_RT[t, s],
#             "P_PN": P_PN[t, s],
#             "P_IN": -lambda_dual[t, s] * S,
#         }
#         data_for_csv.append(row)

# df = pd.DataFrame(data_for_csv)

# output_filename = f"optimization_results_{SEED}.csv"
# df.to_csv(output_filename, index=False, encoding="utf-8-sig")

# print(f"✅ 데이터가 '{output_filename}' 파일로 성공적으로 저장되었습니다.")

,Time,P_DA,P_RT_avg,Lambda,P_PN_avg
0,0,91.140,60.258,182.286,182.286
1,1,80.280,32.637,160.550,160.550
2,2,74.560,25.669,149.110,149.110
3,3,71.640,75.082,150.164,150.164
4,4,71.310,47.337,142.610,142.610
5,5,74.540,33.696,149.084,149.084
6,6,80.820,48.823,128.872,161.642
7,7,87.240,47.280,128.743,174.486
8,8,101.970,38.303,128.743,203.944
9,9,110.370,49.400,128.743,220.740


In [267]:
m5 = gp.Model("DER_Individual_Replay")
# m5.setParam("MIPGap", 1e-5)
m5.setParam(GRB.Param.PoolSearchMode, 1)
m5.setParam(GRB.Param.PoolSolutions, 1)

x = m5.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
yp = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp") ; ym = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym")
dp = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp") ; dm = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm")
z = m5.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z")
zc = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc") ; zd = m5.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")

m5.update()

# NOTE: inefficiency 고려해서 바꿔야함.

obj_lin = (
    gp.quicksum(P_DA[t] * x[i, t] for i in range(I) for t in range(T)) +
    gp.quicksum((1/S) * (
        P_RT[t, s] * yp[i, t, s] - P_PN[t, s] * ym[i, t, s]
    ) for i in range(I) for t in range(T) for s in range(S)) +
    gp.quicksum(
        lambda_rep[t, s] * (dm[i, t, s] - INEFF_INT * dp[i, t, s])
        for i in range(I) for t in range(T) for s in range(S)
    )
)

eps = 1e-8
# quad_reg = gp.quicksum(x[i, t] * x[i, t] for i in range(I) for t in range(T))
quad_reg = gp.quicksum((1 / S) * (dp[i, t, s] * dp[i, t, s] + dm[i, t, s] * dm[i, t, s]) for i in range(I) for t in range(T) for s in range(S))
# quad_reg = gp.quicksum((1 / S) * (dp[i, t, s] * dp[i, t, s]) for i in range(I) for t in range(T) for s in range(S))
# quad_reg = gp.quicksum((1 / S) * (dm[i, t, s] * dm[i, t, s]) for i in range(I) for t in range(T) for s in range(S))

obj = obj_lin - eps * quad_reg

# NOTE
m5.setObjective(obj_lin, GRB.MAXIMIZE)

for i, t, s in product(range(I), range(T), range(S)):
    m5.addConstr(R[i, t, s] - x[i, t] == yp[i, t, s] - ym[i, t, s] + dp[i, t, s] - dm[i, t, s] + zc[i, t, s] - zd[i, t, s])
    m5.addConstr(zd[i, t, s]/INEFF <= z[i, t, s]) ; m5.addConstr(zc[i, t, s]*INEFF <= K[i] - z[i, t, s]) ; m5.addConstr(z[i, t, s] <= K[i])
    m5.addConstr(zd[i, t, s]/INEFF <= DRATE[i]) ; m5.addConstr(zc[i, t, s]*INEFF <= CRATE[i])
    m5.addConstr(z[i, t + 1, s] == z[i, t, s] + INEFF * zc[i, t, s] - zd[i, t, s] / INEFF)
for i, s in product(range(I), range(S)): m5.addConstr(z[i, 0, s] == K0[i])

m5.optimize()

if m5.status == GRB.OPTIMAL:
    num_solutions = m5.SolCount
    print(f"\n--- Solution Pool Analysis ---")
    print(f"Found {num_solutions} solutions in the pool.")

    if num_solutions > 1:
        best_obj = m5.objVal
        print(f"Best objective value: {best_obj:.8f}\n")

        for i in range(num_solutions):
            m5.setParam(GRB.Param.SolutionNumber, i)
            pool_obj = m5.PoolObjVal
            diff = best_obj - pool_obj

            print(f"Solution {i}: Objective = {pool_obj},  Difference from best = {diff}")

    m5.setParam(GRB.Param.SolutionNumber, 0)

    print(f"Optimal solution found! Objective value: {m5.objVal}")
else:
    print("No optimal solution found.")

x_re = np.array([[x[i, t].X for t in range(T)] for i in range(I)])
yp_re = np.array([[[yp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; ym_re = np.array([[[ym[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
dp_re = np.array([[[dp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; dm_re = np.array([[[dm[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
z_re = np.array([[[z[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)])
zc_re = np.array([[[zc[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)]) ; zd_re = np.array([[[zd[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
OBJ_RE = (
    sum(P_DA[t] * x_re[i, t] for i in range(I) for t in range(T))
    + sum(
        (1 / S) * (P_RT[t, s] * yp_re[i, t, s] - P_PN[t, s] * ym_re[i, t, s])
        for i in range(I)
        for t in range(T)
        for s in range(S)
    )
    + sum(
        lambda_rep[t, s]
        * (
            sum(dm_re[i, t, s] for i in range(I))
            - INEFF_INT * sum(dp_re[i, t, s] for i in range(I))
        )
        for t in range(T)
        for s in range(S)
    )
)
QUAD_RE = eps * sum((1 / S) * (dp_re[i, t, s] * dp_re[i, t, s] + dm_re[i, t, s] * dm_re[i, t, s]) for i in range(I) for t in range(T) for s in range(S))

Set parameter PoolSearchMode to value 1
Set parameter PoolSolutions to value 1
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (mac64[arm] - Darwin 25.1.0 25B78)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
PoolSolutions  1
PoolSearchMode  1

Optimize a model with 169000 rows, 169240 columns and 433000 nonzeros
Model fingerprint: 0x23af5399
Coefficient statistics:
  Matrix range     [9e-01, 1e+00]
  Objective range  [3e-02, 1e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e-01, 4e+03]
Presolve removed 100141 rows and 78392 columns
Presolve time: 0.15s
Presolved: 68859 rows, 90848 columns, 309226 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Ordering time: 0.40s

Barrier performed 0 iterations in 0.61 seconds (0.52 work units)
Barrier solve interrupted - model solved by another algorithm


Solved with primal simplex
Iteration    Objective

In [268]:
QUAD_RE

np.float64(0.0332437756356817)

In [269]:
for t, s in product(range(T), range(S)):
    if -lambda_dual[t, s] * S < P_RT[t, s] - 0.00001 or -lambda_dual[t, s] * S > P_PN[t, s] + 0.00001:
        print(f"t={t}, s={s}, P_RT={P_RT[t, s]}, P_IN={-lambda_dual[t, s] * S}, P_PN={P_PN[t, s]}")

t=2, s=9, P_RT=39.405984523033744, P_IN=149.2592592592593, P_PN=149.11
t=2, s=36, P_RT=64.83175048182372, P_IN=149.2592592592593, P_PN=149.11
t=3, s=23, P_RT=45.32513251022292, P_IN=143.42942942942943, P_PN=143.286
t=3, s=43, P_RT=54.410238059953976, P_IN=143.42942942942943, P_PN=143.286
t=3, s=46, P_RT=46.55729250554517, P_IN=143.42942942942943, P_PN=143.286
t=3, s=65, P_RT=56.68180033601745, P_IN=143.42942942942943, P_PN=143.286
t=3, s=68, P_RT=42.78773923512215, P_IN=143.42942942942943, P_PN=143.286
t=3, s=88, P_RT=61.57125016850583, P_IN=143.42942942942943, P_PN=143.286
t=4, s=10, P_RT=65.60886404091826, P_IN=142.7527527527528, P_PN=142.61
t=4, s=17, P_RT=43.99500504375079, P_IN=142.7527527527528, P_PN=142.61
t=4, s=26, P_RT=39.33213934257817, P_IN=142.7527527527528, P_PN=142.61
t=4, s=31, P_RT=50.89218600682328, P_IN=142.7527527527528, P_PN=142.61
t=4, s=95, P_RT=47.359613252710446, P_IN=142.7527527527528, P_PN=142.61
t=5, s=14, P_RT=57.19511256222825, P_IN=149.23323323323325, P_P

In [270]:
print(round(x_ind[:,:].sum(),2), round(x_re[:,:].sum(),2), round(x_hol[:,:].sum(),2))

31138.7 35872.16 35959.61


In [271]:
for i, t, s in product(range(I), range(T), range(S)):
    if dp_re[i, t, s] > 0.001 and dm_re[i, t, s] > 0.001:
        print(f"i={i}, t={t}, s={s}, dp={dp_re[i, t, s]}, dm={dm_re[i, t, s]}")
    if zc_re[i, t, s] > 0.001 and zd_re[i, t, s] > 0.001:
        print(f"i={i}, t={t}, s={s}, zc={zc_re[i, t, s]}, zd={zd_re[i, t, s]}")
    if dp_re[i, t, s] > 0.001 and ym_re[i, t, s] > 0.001:
        print(f"i={i}, t={t}, s={s}, dp={dp_re[i, t, s]}, ym={ym_re[i, t, s]}, P_RT={P_RT[t, s]}, P_IN={-lambda_dual[t, s] * S}, P_PN={P_PN[t, s]}")
    if dm_re[i, t, s] > 0.001 and yp_re[i, t, s] > 0.001:
        print(f"i={i}, t={t}, s={s}, dm={dm_re[i, t, s]}, yp={yp_re[i, t, s]}, P_RT={P_RT[t, s]}, P_IN={-lambda_dual[t, s] * S}, P_PN={P_PN[t, s]}")

In [272]:
header = (f"{'t':>2} | {'R':>8} {'x':>8} {'y+':>8} {'y-':>8} {'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n" + "-" * 90)
print(f"\n[REPLAY] Objective Value = {OBJ_RE:.2f}") ; print(header)
for t in range(T):
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)]) ; x_sum = x_re[:, t].sum()
    yp_avg = np.mean([yp_re[:, t, s].sum() for s in range(S)]) ; ym_avg = np.mean([ym_re[:, t, s].sum() for s in range(S)])
    dp_avg = np.mean([dp_re[:, t, s].sum() for s in range(S)]) ; dm_avg = np.mean([dm_re[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_re[:, t, s].sum() for s in range(S)]) ; zd_avg = np.mean([zd_re[:, t, s].sum() for s in range(S)]) ; z_avg = np.mean([z_re[:, t, s].sum() for s in range(S)])
    print(f"{t:>2} | {R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} {INEFF_INT * dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}")

# print("\n[HOLISTIC]") ; print(header)
# for t in range(T):
#     R_avg = np.mean([R[:, t, s].sum() for s in range(S)]) ; x_sum = x_hol[:, t].sum()
#     yp_avg = np.mean([yp_hol[:, t, s].sum() for s in range(S)]) ; ym_avg = np.mean([ym_hol[:, t, s].sum() for s in range(S)])
#     dp_avg = np.mean([dp_hol[:, t, s].sum() for s in range(S)]) ; dm_avg = np.mean([dm_hol[:, t, s].sum() for s in range(S)])
#     zc_avg = np.mean([zc_hol[:, t, s].sum() for s in range(S)]) ; zd_avg = np.mean([zd_hol[:, t, s].sum() for s in range(S)]) ; z_avg = np.mean([z_hol[:, t, s].sum() for s in range(S)])
#     print(f"{t:>2} | {R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} {dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}")


[REPLAY] Objective Value = 5569728.55
 t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
 0 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00
 1 |     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00     0.00
 2 |     0.00     0.00     0.00     6.67     0.00     0.00     6.67     0.00     0.00
 3 |     0.00     0.00     0.00    35.94     0.00    15.95    51.89     0.00     6.01
 4 |     0.00     0.00     0.00    78.97     0.00    63.62   142.59     0.00    52.71
 5 |     0.00     0.00     0.00     1.38     0.00     0.00     1.38     0.00   181.04
 6 |    56.77     0.00     6.16     0.00    26.82    26.58    50.34     0.00   182.28
 7 |   595.74     0.00    69.76     0.00   149.30    71.66   448.19     0.00   227.59
 8 |  1451.63   778.92    25.28     0.00   225.26   181.81   603.75     0.00   630.96
 9 |  1575

In [273]:
print("="*50) ; print("AGGREGATOR LOSS ANALYSIS") ; print("="*50)
total_losses = []
for t in range(T):
    scenario_losses = []
    for s in range(S):
        total_supply = np.sum(dp_re[:, t, s]) * INEFF_INT 
        total_demand = np.sum(dm_re[:, t, s])
        lambda_price = -lambda_rep[t, s] * S
        loss = (total_demand - total_supply) * (lambda_price - P_RT[t, s])
        scenario_losses.append(loss)
    
    avg_loss = np.mean(scenario_losses)
    total_losses.append(avg_loss)

overall_avg_loss = np.mean(total_losses)
total_loss = np.sum(total_losses)

print("Individual Participation Profit", OBJ_IND)
print("Expected Replay Profit", OBJ_RE)
print(f"Total loss across all time periods: {total_loss:.2f}")
print("Realized Profit", OBJ_RE + total_loss)
print("Holistic Profit", OBJ_HOL)

print(); print("="*60) ; print("INDIVIDUAL PROFIT ANALYSIS") ; print("="*60)
profit_ind = np.zeros(I) ; profit_re = np.zeros(I) ; profit_hol = np.zeros(I) ; profit_re_adjusted = np.zeros(I)

price_weighted_usage = np.zeros(I)
for i, t, s in product(range(I), range(T), range(S)):
        lambda_price = -lambda_rep[t, s] * S
        dm_contribution = dm_re[i, t, s] * (lambda_price)
        dp_contribution = dp_re[i, t, s] * INEFF_INT * (lambda_price) 
        price_weighted_usage[i] += dm_contribution + dp_contribution

total_price_weighted_usage = np.sum(price_weighted_usage)
print(f"Total price-weighted usage: {total_price_weighted_usage:.2f}")

for i in range(I):
    # 1. Individual Case (변경 없음)
    profit_ind[i] = 0
    for t in range(T):
        profit_ind[i] += P_DA[t] * x_ind[i, t]
        profit_ind[i] += np.mean([P_RT[t, s] * yp_ind[i, t, s] for s in range(S)])
        profit_ind[i] -= np.mean([P_PN[t, s] * ym_ind[i, t, s] for s in range(S)])
    
    # 2. Replay Case (Decomposition)
    profit_re[i] = 0
    for t in range(T):
        profit_re[i] += P_DA[t] * x_re[i, t]
        profit_re[i] += np.mean([P_RT[t, s] * yp_re[i, t, s] for s in range(S)])
        profit_re[i] -= np.mean([P_PN[t, s] * ym_re[i, t, s] for s in range(S)])
        
        lambda_price = -lambda_rep[t, :] * S
        # [수정 3] 내부 판매 수익 계산 시 효율(INEFF_INT) 반영
        profit_re[i] += np.mean([lambda_price[s] * dp_re[i, t, s] * INEFF_INT for s in range(S)]) 
        profit_re[i] -= np.mean([lambda_price[s] * dm_re[i, t, s] for s in range(S)])
    
    # 3. Holistic Case (Centralized)
    profit_hol[i] = 0
    for t in range(T):
        profit_hol[i] += P_DA[t] * x_hol[i, t]
        profit_hol[i] += np.mean([P_RT[t, s] * yp_hol[i, t, s] for s in range(S)])
        profit_hol[i] -= np.mean([P_PN[t, s] * ym_hol[i, t, s] for s in range(S)])
        
        lambda_price = -lambda_rep[t, :] * S
        # [수정 4] Holistic 비교 시에도 동일하게 효율 반영
        profit_hol[i] += np.mean([lambda_price[s] * dp_hol[i, t, s] * INEFF_INT for s in range(S)])
        profit_hol[i] -= np.mean([lambda_price[s] * dm_hol[i, t, s] for s in range(S)])

# 손실 배분 및 최종 이익 계산
loss_per_player = np.zeros(I)
for i in range(I):
    if total_price_weighted_usage > 0:
        loss_per_player[i] = total_loss * (price_weighted_usage[i] / total_price_weighted_usage)
    else:
        loss_per_player[i] = total_loss / I  
    profit_re_adjusted[i] = profit_re[i] + loss_per_player[i]

print(f"{'Player':<8} {'Individual':<12} {'Replay':<12} {'Re+Loss':<12} {'Holistic':<12} {'Price Weight':<12} {'Loss Share':<12} {'Re+Loss-Ind (%)':<22}") 
print("-" * 125)

total_ind = 0 ; total_re = 0 ; total_hol = 0 ; total_re_adj = 0

for i in range(I):
    diff_adj_ind = profit_re_adjusted[i] - profit_ind[i]
    if profit_ind[i] != 0:
        percentage_change = (diff_adj_ind / profit_ind[i]) * 100
        final_column_str = f"{diff_adj_ind:<12.2f} ({percentage_change:+.1f}%)"
    else:
        final_column_str = f"{diff_adj_ind:<12.2f} (N/A)"

    print(f"{i:<8} {profit_ind[i]:<12.2f} {profit_re[i]:<12.2f} {profit_re_adjusted[i]:<12.2f} {profit_hol[i]:<12.2f} {loss_per_player[i]:<12.2f} {final_column_str:<22}")
    
    total_ind += profit_ind[i]
    total_re += profit_re[i]
    total_hol += profit_hol[i]
    total_re_adj += profit_re_adjusted[i]

total_diff_adj_ind = total_re_adj - total_ind
if total_ind != 0:
    total_percentage_change = (total_diff_adj_ind / total_ind) * 100
    total_final_column_str = f"{total_diff_adj_ind:<12.2f} ({total_percentage_change:+.1f}%)"
else:
    total_final_column_str = f"{total_diff_adj_ind:<12.2f} (N/A)"
    
print("-" * 125)
print(f"{'TOTAL':<8} {total_ind:<12.2f} {total_re:<12.2f} {total_re_adj:<12.2f} {total_hol:<12.2f} {np.sum(loss_per_player):<12.2f} {total_final_column_str:<22}")

# =======================================
# ⬇️ DataFrame으로 정리
# =======================================
results = []
for i in range(I):
    diff_adj_ind = profit_re_adjusted[i] - profit_ind[i]
    if profit_ind[i] != 0:
        percentage_change = (diff_adj_ind / profit_ind[i]) * 100
    else:
        percentage_change = np.nan  # 0 나눗셈 방지

    results.append({
        "Player": i,
        "Individual Profit": profit_ind[i],
        "Replay Profit": profit_re[i],
        "Replay + Loss": profit_re_adjusted[i],
        "Holistic Profit": profit_hol[i],
        "Loss Share": loss_per_player[i],
        "Replay+Loss - Individual": diff_adj_ind,
        "Change (%)": percentage_change
    })

results.append({
    "Player": "TOTAL",
    "Individual Profit": np.sum(profit_ind),
    "Replay Profit": np.sum(profit_re),
    "Replay + Loss": np.sum(profit_re_adjusted),
    "Holistic Profit": np.sum(profit_hol),
    "Loss Share": np.sum(loss_per_player),
    "Replay+Loss - Individual": np.sum(profit_re_adjusted) - np.sum(profit_ind),
    "Change (%)": (np.sum(profit_re_adjusted) - np.sum(profit_ind)) / np.sum(profit_ind) * 100 if np.sum(profit_ind) != 0 else np.nan
})

df_result = pd.DataFrame(results)

# =======================================
# ⬇️ CSV 저장 경로 설정
# =======================================
# output_dir = r"C:\Users\jangseohyun\SynologyDrive\workspace\symply\DER\solution"
# os.makedirs(output_dir, exist_ok=True)
# output_path = os.path.join(output_dir, f"profit_analysis_{SEED}.csv")

# df_result.to_csv(output_path, index=False, encoding="utf-8-sig")
# print(f"\n✅ 결과가 CSV로 저장되었습니다: {output_path}")

AGGREGATOR LOSS ANALYSIS
Individual Participation Profit 4863178.383748347
Expected Replay Profit 5569728.553776862
Total loss across all time periods: -18094.24
Realized Profit 5551634.316897705
Holistic Profit 5569728.55377686

INDIVIDUAL PROFIT ANALYSIS
Total price-weighted usage: 135898331.78
Player   Individual   Replay       Re+Loss      Holistic     Price Weight Loss Share   Re+Loss-Ind (%)       
-----------------------------------------------------------------------------------------------------------------------------
0        343778.18    393633.19    392267.53    393633.19    -1365.66     48489.35     (+14.1%) 
1        399199.52    457640.53    456295.70    457640.53    -1344.83     57096.18     (+14.3%) 
2        582329.13    669396.29    667141.10    669396.29    -2255.19     84811.97     (+14.6%) 
3        885347.98    1003795.94   1000586.99   1003795.94   -3208.94     115239.01    (+13.0%) 
4        211822.01    263027.91    261991.86    263027.91    -1036.04     5016